In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
"""
==============================================================================
Adapted DINOv2 (ABIDE-Tuned) Architecture Pipeline
==============================================================================
Architecture Flow:
ABIDE MRI -> DINOv2 Backbone -> Residual Adapters -> Multi-Level Feature Fusion
          -> Attention Pooling -> Prototype Learning Head -> Explainability -> ASD / Control
"""

import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
for _p in ["timm", "nibabel", "SimpleITK", "tqdm", "shap"]:
    try:
        pip(_p)
    except Exception:
        pass

import os, math, random, warnings, logging, json, hashlib
from pathlib import Path
from copy import deepcopy
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
log = logging.getLogger("Adapted-DINOv2")

import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

import timm
import nibabel as nib
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             roc_auc_score)

try:
    import SimpleITK as sitk
    HAVE_SITK = True
except Exception:
    HAVE_SITK = False

try:
    import shap
    HAVE_SHAP = True
except Exception:
    HAVE_SHAP = False

# =============================================================================
# GLOBAL CONFIG
# =============================================================================
SEED   = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ON_GPU = DEVICE.type == "cuda"

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()

CONFIG = {
    "model_type": "adapted_dinov2", 
    "dinov2_variant": "vit_base_patch14_reg4_dinov2.lvd142m",
    "img_size": 224,
    "unfreeze_last_n_blocks": 2,
    "adapter_dim": 64,
    "dropout": 0.3,
    "mni_template_path": None,
    "atlas_path": None,
    "auto_fetch_mni": True,
    "cache_mni_registration": True,
    "lr_backbone": 2e-5,
    "lr_new_modules": 5e-4,
    "weight_decay": 1e-3,
    "batch_size": 16,
    "epochs": 100,
    "early_stop_patience": 25,
    "num_workers": 2 if ON_GPU else 0,
    "n_explain_samples": 6,
}

OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LABEL_MAP = {
    "asd": 1, "autism": 1, "autistic": 1,
    "tc": 0, "control": 0, "typical": 0, "healthy": 0, "normal": 0, "hc": 0,
}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".nii", ".gz"}

# =============================================================================
# DATASET DISCOVERY & PREPROCESSING
# =============================================================================
def auto_detect_dataset() -> dict:
    kaggle_in = Path("/kaggle/input")
    if not kaggle_in.exists() or not any(kaggle_in.iterdir()):
        raise FileNotFoundError("ERROR: /kaggle/input is empty! Attach your dataset.")

    for ds in sorted(kaggle_in.iterdir()):
        if not ds.is_dir(): continue
        fmri_root, smri_root = ds / "fmri", ds / "smri"
        if fmri_root.exists() and smri_root.exists():
            smri, fmri = _collect_nii(smri_root), _collect_nii(fmri_root)
            if smri["asd"] and smri["ctl"] and fmri["asd"] and fmri["ctl"]:
                return {"mode": "dual", "smri": smri, "fmri": fmri, "root": ds}

    for ds in sorted(kaggle_in.iterdir()):
        if not ds.is_dir(): continue
        found = _collect_images(ds)
        if found["asd"] and found["ctl"]:
            return {"mode": "single", "single": found, "root": ds}

    raise FileNotFoundError("ERROR: No valid dataset structure matching single or dual modalities found.")

def _label_from_path(p: Path):
    for parent in p.parents:
        key = parent.name.lower()
        for kw, lbl in LABEL_MAP.items():
            if kw in key: return lbl
    return None

def _collect_nii(root: Path) -> dict:
    result = {"asd": [], "ctl": []}
    for p in sorted(root.rglob("*")):
        sfx = p.suffix.lower()
        if sfx == ".gz" and p.stem.endswith(".nii"): sfx = ".nii"
        if sfx != ".nii": continue
        lbl = _label_from_path(p)
        if lbl == 1: result["asd"].append(str(p))
        elif lbl == 0: result["ctl"].append(str(p))
    return result

def _collect_images(root: Path) -> dict:
    result = {"asd": [], "ctl": []}
    for p in sorted(root.rglob("*")):
        sfx = p.suffix.lower()
        if sfx == ".gz" and p.stem.endswith(".nii"): sfx = ".nii"
        if sfx not in IMG_EXTS: continue
        lbl = _label_from_path(p)
        if lbl == 1: result["asd"].append(str(p))
        elif lbl == 0: result["ctl"].append(str(p))
    return result

def dataset_to_samples(info: dict):
    def make_samples(d):
        s = [(p, 1) for p in d["asd"]] + [(p, 0) for p in d["ctl"]]
        random.shuffle(s)
        return s
    mode = info["mode"]
    if mode == "single":
        return make_samples(info["single"]), [], mode
    smri, fmri = make_samples(info["smri"]), make_samples(info["fmri"])
    smri.sort(key=lambda x: x[1], reverse=True)
    fmri.sort(key=lambda x: x[1], reverse=True)
    n = min(len(smri), len(fmri))
    return smri[:n], fmri[:n], mode

def fcm_norm(vol: np.ndarray) -> np.ndarray:
    try:
        import skfuzzy as fuzz
        flat = vol.flatten().astype(np.float64).reshape(1, -1)
        cntr, *_ = fuzz.cmeans(flat, c=3, m=2.0, error=0.005, maxiter=500)
        ref = cntr[np.argmax(cntr[:, 0]), 0]
        if ref > 0: vol = vol / ref * 128.
    except Exception:
        mu, sd = vol.mean(), vol.std()
        if sd > 0: vol = (vol - mu) / sd * 64 + 128
    return vol.clip(0, 255)

def minmax(sl: np.ndarray) -> np.ndarray:
    mn, mx = sl.min(), sl.max()
    if mx == mn: return np.zeros_like(sl, dtype=np.uint8)
    return ((sl - mn) / (mx - mn) * 255).astype(np.uint8)

def load_as_rgb(path_or_arr, cfg=CONFIG) -> np.ndarray:
    img_size = cfg["img_size"]
    if isinstance(path_or_arr, np.ndarray):
        if path_or_arr.ndim == 2:
            return np.stack([path_or_arr] * 3, axis=-1).astype(np.uint8)
        elif path_or_arr.ndim == 3 and path_or_arr.shape[-1] != 3:
            return np.stack([path_or_arr[:, :, 0]] * 3, axis=-1).astype(np.uint8)
        return path_or_arr.astype(np.uint8)

    p = Path(path_or_arr)
    sfx = p.suffix.lower()

    if sfx in (".nii", ".gz"):
        data = nib.load(str(p)).get_fdata()
        if data.ndim == 4: data = data.mean(axis=-1)
        data = fcm_norm(data)

        if data.ndim == 2:
            sl = minmax(data)
            pil = Image.fromarray(sl).resize((img_size, img_size), Image.BILINEAR)
            arr = np.array(pil)
            return np.stack([arr, arr, arr], axis=-1).astype(np.uint8)

        x, y, z = data.shape[:3]
        planes = [data[x // 2, :, :], data[:, y // 2, :], data[:, :, z // 2]]
        chs = []
        for pl in planes:
            sl = minmax(pl)
            pil = Image.fromarray(sl).resize((img_size, img_size), Image.BILINEAR)
            chs.append(np.array(pil))
        return np.stack(chs, axis=-1).astype(np.uint8)

    img = Image.open(str(p)).convert("RGB")
    img = img.resize((img_size, img_size), Image.BILINEAR)
    return np.array(img)

TRAIN_TF = T.Compose([
    T.ToPILImage(),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
EVAL_TF = T.Compose([
    T.ToPILImage(), T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class BrainDataset(Dataset):
    def __init__(self, samples, augment=False, cfg=CONFIG):
        self.samples = samples
        self.tf = TRAIN_TF if augment else EVAL_TF
        self.imgs = [load_as_rgb(p, cfg) for p, _ in tqdm(samples, leave=False, desc="Loading Data")]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        _, lbl = self.samples[idx]
        return self.tf(self.imgs[idx]), torch.tensor(lbl, dtype=torch.long)

# =============================================================================
# ARCHITECTURE MODULES
# =============================================================================

# --- Residual Bottleneck Adapter ---
class ResidualAdapter(nn.Module):
    """
    Input (768) -> LayerNorm -> Linear(768 -> 64) -> GELU 
                -> Dropout -> Linear(64 -> 768) -> Residual Addition
    """
    def __init__(self, in_dim=768, bottleneck_dim=64, dropout=0.1):
        super().__init__()
        self.adapter = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, bottleneck_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(bottleneck_dim, in_dim)
        )
    def forward(self, x):
        return x + self.adapter(x)

# --- Multi-Level Feature Fusion ---
class MultiLevelFeatureFusion(nn.Module):
    """Combines adapter-enhanced tokens from multiple transformer depth levels."""
    def __init__(self, embed_dim, num_levels):
        super().__init__()
        self.weights = nn.Parameter(torch.ones(num_levels) / num_levels)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, feature_list):
        # feature_list contains [B, N, D] tensors
        stacked = torch.stack(feature_list, dim=0) # [L, B, N, D]
        w = F.softmax(self.weights, dim=0).view(-1, 1, 1, 1)
        fused = torch.sum(stacked * w, dim=0)
        return self.norm(fused)

# --- Attention Pooling ---
class AttentionPooling(nn.Module):
    """Learned attention query over patch tokens replacing naive CLS selection."""
    def __init__(self, embed_dim):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.scale = embed_dim ** -0.5

    def forward(self, x): # x: [B, N, D]
        keys = self.key_proj(x)
        attn_scores = torch.matmul(self.query, keys.transpose(-2, -1)) * self.scale
        attn_weights = F.softmax(attn_scores, dim=-1) # [B, 1, N]
        pooled = torch.matmul(attn_weights, x).squeeze(1) # [B, D]
        return pooled, attn_weights

# --- Prototype Learning Head ---
class PrototypeHead(nn.Module):
    """Classifies feature vectors based on cosine distance to learned category prototypes."""
    def __init__(self, in_dim, num_classes=2, temp=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.temp = temp
        self.prototypes = nn.Parameter(torch.randn(num_classes, in_dim))
        nn.init.orthogonal_(self.prototypes)

    def forward(self, x):
        x_norm = F.normalize(x, p=2, dim=-1)
        proto_norm = F.normalize(self.prototypes, p=2, dim=-1)
        cosine_sim = torch.matmul(x_norm, proto_norm.T) # [B, num_classes]
        logits = cosine_sim / self.temp
        return logits

# =============================================================================
# FULL ARCHITECTURE INTEGRATION
# =============================================================================
class AdaptedDINOv2(nn.Module):
    def __init__(self, cfg=CONFIG, num_classes=2):
        super().__init__()
        self.cfg = cfg
        
        # 1. DINOv2 Backbone
        log.info(f"[Architecture] Loading Backbone: {cfg['dinov2_variant']}")
        self.backbone = timm.create_model(
            cfg["dinov2_variant"], pretrained=True, num_classes=0, img_size=cfg["img_size"]
        )
        for p in self.backbone.parameters():
            p.requires_grad = False

        embed_dim = self.backbone.num_features
        num_blocks = len(self.backbone.blocks)
        
        # Unfreeze specified number of last transformer blocks
        unfreeze_n = cfg["unfreeze_last_n_blocks"]
        for blk in list(self.backbone.blocks.children())[-unfreeze_n:]:
            for p in blk.parameters():
                p.requires_grad = True

        # 2. Residual Adapters attached to trailing layers
        self.adapter_indices = list(range(num_blocks - unfreeze_n, num_blocks))
        self.adapters = nn.ModuleDict({
            str(idx): ResidualAdapter(embed_dim, cfg["adapter_dim"], cfg["dropout"])
            for idx in self.adapter_indices
        })

        # 3. Multi-Level Feature Fusion
        self.fusion = MultiLevelFeatureFusion(embed_dim, len(self.adapter_indices))

        # 4. Attention Pooling
        self.att_pool = AttentionPooling(embed_dim)

        # 5. Prototype Learning Head
        self.head = PrototypeHead(embed_dim, num_classes=num_classes)

    def forward(self, x, return_maps=False):
        # Extract features manually through blocks to inject adapters
        tokens = self.backbone.patch_embed(x)
        tokens = self.backbone._pos_embed(tokens)
        if hasattr(self.backbone, "norm_pre"):
            tokens = self.backbone.norm_pre(tokens)

        intermediate_feats = []
        attentions = []

        for idx, blk in enumerate(self.backbone.blocks):
            tokens = blk(tokens)
            if idx in self.adapter_indices:
                tokens = self.adapters[str(idx)](tokens)
                intermediate_feats.append(tokens)

        # Multi-Level Feature Fusion
        fused_tokens = self.fusion(intermediate_feats)

        # Attention Pooling
        pooled_feat, pool_attn = self.att_pool(fused_tokens)

        # Prototype Head
        logits = self.head(pooled_feat)

        if return_maps:
            return logits, pool_attn, fused_tokens
        return logits

# =============================================================================
# MODULE 8: OPTIMIZER & TRAINING PIPELINE
# =============================================================================
def build_optimizer(model: nn.Module, cfg=CONFIG):
    backbone_params, new_module_params = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad: continue
        if "backbone" in name:
            backbone_params.append(p)
        else:
            new_module_params.append(p)

    groups = [
        {"params": backbone_params, "lr": cfg.get("lr_backbone", 2e-5), "weight_decay": cfg["weight_decay"]},
        {"params": new_module_params, "lr": cfg.get("lr_new_modules", 5e-4), "weight_decay": cfg["weight_decay"]}
    ]
    return optim.AdamW(groups)

class EarlyStopping:
    def __init__(self, patience=25):
        self.patience = patience; self.counter = 0; self.best = None; self.stop = False; self.state = None
    def __call__(self, loss, model):
        if self.best is None or loss < self.best - 1e-4:
            self.best = loss; self.counter = 0
            self.state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            self.counter += 1
            if self.counter >= self.patience: self.stop = True

def train_epoch(model, loader, crit, opt):
    model.train(); total = 0.; n = 0
    for X, y in loader:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad()
        loss = crit(model(X), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item() * len(y); n += len(y)
    return total / max(n, 1)

@torch.no_grad()
def eval_epoch(model, loader, crit):
    model.eval(); total = 0.; preds = []; trues = []; probs = []; n = 0
    for X, y in loader:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        lg = model(X); total += crit(lg, y).item() * len(y); n += len(y)
        preds.extend(lg.argmax(1).cpu().numpy())
        trues.extend(y.cpu().numpy())
        probs.extend(F.softmax(lg, 1)[:, 1].float().cpu().numpy())
    auc = roc_auc_score(trues, probs) if len(set(trues)) > 1 else 0.
    return total / max(n, 1), accuracy_score(trues, preds) * 100, f1_score(trues, preds, zero_division=0) * 100, auc, preds, trues, probs

def train_model(model, tr_ld, vl_ld, cfg=CONFIG, verbose=True):
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    opt = build_optimizer(model, cfg)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"], eta_min=1e-6)
    es = EarlyStopping(patience=cfg["early_stop_patience"])

    for ep in range(1, cfg["epochs"] + 1):
        tl = train_epoch(model, tr_ld, crit, opt)
        vl, acc, f1, auc, *_ = eval_epoch(model, vl_ld, crit)
        sched.step()
        if verbose:
            print(f"Epoch {ep:02d}/{cfg['epochs']} | Train Loss: {tl:.4f} | Val Loss: {vl:.4f} | Acc: {acc:.2f}% | F1: {f1:.2f}%")
        es(vl, model)
        if es.stop:
            if verbose: log.info(f"Early stop triggered at epoch {ep}")
            break

    if es.state: model.load_state_dict(es.state)
    return model

# =============================================================================
# MODULE 9: ADVANCED EXPLAINABILITY (Grad-CAM, Attention Rollout, SHAP)
# =============================================================================
class IntegratedExplainer:
    def __init__(self, model):
        self.model = model

    def get_attention_rollout(self, img_tensor):
        """Generates visual spatial maps from Attention Pooling weights."""
        self.model.eval()
        with torch.no_grad():
            _, attn_weights, _ = self.model(img_tensor.unsqueeze(0).to(DEVICE), return_maps=True)
        
        # Omit prefix/CLS tokens if present and format as spatial grid
        patches_attn = attn_weights[0, 0, self.model.backbone.num_prefix_tokens:]
        side = int(math.sqrt(patches_attn.shape[0]))
        amap = patches_attn.reshape(side, side).cpu().numpy()
        amap = cv2.resize(amap, (CONFIG["img_size"], CONFIG["img_size"]), interpolation=cv2.INTER_CUBIC)
        return (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)

    def get_gradcam(self, img_tensor, target_class=1):
        """Grad-CAM computed over fused transformer tokens."""
        self.model.eval()
        x = img_tensor.unsqueeze(0).to(DEVICE).requires_grad_()
        logits, _, fused_tokens = self.model(x, return_maps=True)
        
        score = logits[0, target_class]
        score.backward()

        gradients = x.grad.data[0].cpu().numpy()
        cam = np.mean(np.maximum(gradients, 0), axis=0)
        cam = cv2.resize(cam, (CONFIG["img_size"], CONFIG["img_size"]))
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    def run_shap_summary(self, val_loader, n_samples=10):
        """Generates SHAP summary outputs if SHAP library is present."""
        if not HAVE_SHAP:
            log.warning("SHAP not installed. Skipping SHAP analysis.")
            return
        
        self.model.eval()
        imgs, _ = next(iter(val_loader))
        imgs = imgs[:n_samples].to(DEVICE)
        
        explainer = shap.GradientExplainer(self.model, imgs)
        shap_values = explainer.shap_values(imgs)
        
        log.info("[Module 9] SHAP explainer analysis completed.")
        return shap_values

def generate_explanations(model, vl_ds, output_dir=OUTPUT_DIR):
    explainer = IntegratedExplainer(model)
    n = min(CONFIG["n_explain_samples"], len(vl_ds))
    fig, axes = plt.subplots(n, 3, figsize=(10, 3.2 * n))

    for row in range(n):
        img_rgb = vl_ds.imgs[row]
        _, label = vl_ds.samples[row]
        tensor = EVAL_TF(img_rgb)

        att_map = explainer.get_attention_rollout(tensor)
        cam_map = explainer.get_gradcam(tensor, target_class=label)

        orig_resized = cv2.resize(img_rgb, (CONFIG["img_size"], CONFIG["img_size"]))
        label_str = "ASD" if label == 1 else "Control"

        axes[row, 0].imshow(orig_resized); axes[row, 0].set_title(f"Original ({label_str})")
        axes[row, 1].imshow(att_map, cmap="inferno"); axes[row, 1].set_title("Attention Rollout")
        axes[row, 2].imshow(cam_map, cmap="jet"); axes[row, 2].set_title("Grad-CAM")
        
        for c in range(3): axes[row, c].axis("off")

    plt.tight_layout()
    save_path = os.path.join(output_dir, "explainability_outputs.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    log.info(f"[Module 9] Saved explainability visual outputs -> {save_path}")

# =============================================================================
# MAIN EXECUTION PIPELINE
# =============================================================================
def main():
    log.info("\n[Setup] Auto-detecting ABIDE dataset...")
    info = auto_detect_dataset()
    smri_samples, _, mode = dataset_to_samples(info)

    tr_s, vl_s = train_test_split(
        smri_samples, test_size=0.2, stratify=[l for _, l in smri_samples], random_state=SEED
    )

    log.info("[Setup] Building DataLoaders...")
    tr_ds, vl_ds = BrainDataset(tr_s, augment=True), BrainDataset(vl_s, augment=False)
    tr_ld = DataLoader(tr_ds, CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"], pin_memory=ON_GPU)
    vl_ld = DataLoader(vl_ds, CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=ON_GPU)

    log.info("\n[Architecture] Initializing Adapted DINOv2 Pipeline...")
    model = AdaptedDINOv2(cfg=CONFIG).to(DEVICE)

    log.info("\n[Module 8] Training Model...")
    model = train_model(model, tr_ld, vl_ld, cfg=CONFIG)

    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    _, acc, f1, auc, preds, trues, _ = eval_epoch(model, vl_ld, crit)

    print(f"\n{'='*60}\n FINAL ADAPTED DINOv2 EVALUATION\n{'='*60}")
    print(classification_report(trues, preds, target_names=["Control", "ASD"]))
    print(f"Accuracy: {acc:.2f}% | F1-Score: {f1:.2f}% | ROC-AUC: {auc:.4f}")

    log.info("\n[Module 9] Generating Explainability Outputs...")
    try:
        generate_explanations(model, vl_ds)
    except Exception as e:
        log.warning(f"Skipped explainability due to error: {e}")

    ckpt_path = os.path.join(OUTPUT_DIR, "adapted_dinov2_best.pt")
    torch.save(model.state_dict(), ckpt_path)
    log.info(f"Model saved successfully to {ckpt_path}")

if __name__ == "__main__":
    main()
